In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import ticker
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

from numerize.numerize import numerize
import os

In [ ]:
con=sqlite3.connect('../data/zephyr.sqlite')

In [ ]:
cursor = con.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

print(cursor.fetchall())

In [ ]:
cursor.execute("SELECT * FROM metadata;")
names = list(map(lambda x: x[0], cursor.description))
names

In [ ]:
df = pd.read_sql_query("SELECT * FROM t ;",con)

df_src=df.loc[(df['Language']=='C/C++ Header') |
       (df['Language']=='C') |
       (df['Language']=='C++') |
       (df['Language']=='Assembly') 
      ].copy()

#newer versions of cloc add an id column which we don't need
df_src.drop(['id'], axis=1, inplace=True)

df_src['base_dir']=df_src['File_dirname'].apply( lambda x: (x.split("/")[0]))

df_src

In [ ]:
df_counts=df_src.groupby(['Project','base_dir']).sum(numeric_only=True).copy()
df_counts

In [ ]:
df_counts.drop(['nBlank','nComment','nScaled'],axis=1, inplace=True)

In [ ]:
df_counts_pivot=df_counts.pivot_table(index="Project",columns="base_dir", fill_value=0 )
df_counts_pivot

In [ ]:
#only use patch level 0
df_plot=df_counts_pivot.filter(regex="\\.0$", axis=0).copy()

In [ ]:
df_plot.rename(index={'v1.0.0':'v1.00.0',
                      'v1.1.0':'v1.01.0',
                      'v1.2.0':'v1.02.0',
                      'v1.3.0':'v1.03.0',
                      'v1.4.0':'v1.04.0',
                      'v1.5.0':'v1.05.0',
                      'v1.6.0':'v1.06.0',
                      'v1.7.0':'v1.07.0',
                      'v1.8.0':'v1.08.0',
                      'v1.9.0':'v1.09.0',
                     
                     }, inplace=True)

In [ ]:
df_plot.sort_index(inplace=True)
df_plot

In [ ]:
df_plot.columns=[c[1] for c in df_plot.columns.values]

In [ ]:
# Choose some nice levels
plt.rcParams["figure.figsize"] = [16, 9]
plt.rcParams["figure.autolayout"] = True

legend=['arch','boards','kernel','soc','modules','lib','include','subsys','drivers', 'samples', 'tests', 'net' ]
#legend=['samples' ]
legend=sorted(legend)

colors = ["darkorange", "gold", "lawngreen", "lightseagreen"]
cmap1 = LinearSegmentedColormap.from_list("mycmap", colors)

colormap = mpl.colormaps["tab20"](np.arange(len(legend)))
colormap = mpl.colormaps['viridis'].resampled(len(legend)).colors
colormap = cmap1.resampled(len(legend))(np.arange(len(legend)))

fig, ax = plt.subplots(figsize=(16,9), constrained_layout=True)
ax.stackplot(df_plot.index, [df_plot[name] for name in legend],
             labels=legend, colors=colormap
            )
ax.set_ylabel("Lines of Code")             
ax.yaxis.set_major_formatter(lambda x,pos: f"{numerize(x)}")
ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(),rotation=45)
ax.legend(loc='upper left')
             
fig.savefig('../figures/zephyr-growth.svg', format='svg')     

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

viridis = mpl.colormaps['viridis'].resampled(8)
viridis.colors

In [ ]:
mpl.colormaps["tab20"](np.arange(12))

In [ ]:
colors = ["darkorange", "gold", "lawngreen", "lightseagreen"]
cmap1 = LinearSegmentedColormap.from_list("mycmap", colors)

In [ ]:
cmap1.resampled(12)(np.arange(12))